# Feature Enggineering- Creating new clinical feature


### 🎯 Introduction
This notebook focuses exclusively on constructing new engineered features based on the empirical insights and non-linear risk spikes discovered during our Exploratory Data Analysis (EDA). To provide our machine learning models with strong clinical signals, we transform the raw patient data into four advanced domain-specific features:

1. **`PULSE_PRESSURE`**: Calculated as Systolic minus Diastolic blood pressure to act as a direct proxy for vascular stiffness and cardiac workload.
2. **`INFLAMMATORY_INDEX`**: A continuous interaction term multiplying White Blood Cells by Erythrocyte Sedimentation Rate to track early acute inflammatory storms.
3. **`HIGH_RISK_PROFILE`**: A strict binary indicator capturing highly vulnerable elderly patients based on age, admission severity, and time delays.
4. **`ELDERLY_HYPERTENSION_RISK`**: A continuous interaction term multiplying age by hypertension grade to capture the combined risk spike in older populations.

By introducing these specialized variables, we ensure our downstream classifiers can effectively detect hidden patterns and predict critical complications before they happen.


In [ ]:
# Core Imports
import os
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Step 1: Loading data from drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder_drive = "/content/drive/MyDrive/Mid-term Project/"

# Loading raw sliced features with strict index reconstruction (index_col=0)
X_train_d1 = pd.read_csv(os.path.join(folder_drive, "X_train_day1_raw_sliced.csv"), index_col=0)
X_train_d2 = pd.read_csv(os.path.join(folder_drive, "X_train_day2_raw_sliced.csv"), index_col=0)
X_train_d3 = pd.read_csv(os.path.join(folder_drive, "X_train_day3_raw_sliced.csv"), index_col=0)

print("✅ Raw sliced feature matrices loaded successfully with safe patient index alignment!")
print(f"Day 1 Shape: {X_train_d1.shape}")
print(f"Day 2 Shape: {X_train_d2.shape}")
print(f"Day 3 Shape: {X_train_d3.shape}")


✅ Raw sliced feature matrices loaded successfully with safe patient index alignment!
Day 1 Shape: (1360, 46)
Day 2 Shape: (1360, 58)
Day 3 Shape: (1360, 107)


# Step 2: Registering the Dynamic Preprocessing Pipeline (Imputation, Encoding, and Scaling)

In [ ]:
# Global dictionary to quietly store transformers without altering function output type
all_day_transformers = {}


def process_day_data(X_day, day_key="default"):
    """Processes the input day dataframe by applying median/mode imputation,

    one-hot encoding for multi-class variables, and continuous scaling, while
    safely preserving the patient index alignment and avoiding data leakage.
    """
    X_day = (
        X_day.copy()
    )  # Deep copy to prevent Side-Effect / SettingWithCopyWarning
    fitted_objects = {}

    # 1. Automatic classification based on data types and unique value thresholds
    numeric_cols = [
        c
        for c in X_day.columns
        if X_day[c].dtype in ["float64", "int64"] and X_day[c].nunique() > 10
    ]
    categorical_cols = [c for c in X_day.columns if c not in numeric_cols]

    # 2. Handle missing values (Imputation) using clinical strategies
    num_imputer = SimpleImputer(strategy="median")
    if numeric_cols:
        X_day[numeric_cols] = num_imputer.fit_transform(X_day[numeric_cols])

    cat_imputer = SimpleImputer(strategy="most_frequent")
    if categorical_cols:
        X_day[categorical_cols] = cat_imputer.fit_transform(
            X_day[categorical_cols]
        )

    # 3. Apply One-Hot Encoding strictly to true multi-class variables (>2 unique categories)
    true_cat_cols = [c for c in categorical_cols if X_day[c].nunique() > 2]
    if true_cat_cols:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        encoded_array = encoder.fit_transform(X_day[true_cat_cols])

        # CRITICAL FIX: Lock the encoded dataframe index to the original matrix state
        encoded_df = pd.DataFrame(
            encoded_array,
            columns=encoder.get_feature_names_out(true_cat_cols),
            index=X_day.index,  # Secures patient row matching
        )
        # Drop categorical columns cleanly without destroying row indexes
        X_day = X_day.drop(columns=true_cat_cols)
        X_day = pd.concat([X_day, encoded_df], axis=1)
        fitted_objects["encoder"] = encoder

    # 4. Normalize continuous features using StandardScaler (Mean=0, SD=1)
    scaler = StandardScaler()
    if numeric_cols:
        X_day[numeric_cols] = scaler.fit_transform(X_day[numeric_cols])

    # 5. Background state persistence: Storing learned metadata for future Test Set transformation
    fitted_objects["num_imputer"] = num_imputer
    fitted_objects["cat_imputer"] = cat_imputer
    fitted_objects["scaler"] = scaler
    fitted_objects["numeric_cols"] = numeric_cols
    fitted_objects["categorical_cols"] = categorical_cols
    fitted_objects["true_cat_cols"] = true_cat_cols
    all_day_transformers[day_key] = fitted_objects

    # Returns ONLY the processed DataFrame to protect calling cells from format exceptions
    return X_day


print(
    "✅ 'process_day_data' pipeline function successfully defined in memory with Index and Leakage safety."
)


✅ 'process_day_data' pipeline function successfully defined in memory with Index and Leakage safety.


# Step 3: Create independent copies

In [ ]:
# Create independent deep copies to isolate features in memory and prevent tracking bugs
X_train_d1_fe = X_train_d1.copy()
X_train_d2_fe = X_train_d2.copy()
X_train_d3_fe = X_train_d3.copy()

print("✅ Fresh isolated copies initialized in memory. Ready for independent feature engineering.")


✅ Fresh isolated copies initialized in memory. Ready for independent feature engineering.


# Step 4: Feature 1- Pulse Pressure

- **PULSE_PRESSURE**: The difference between systolic and diastolic blood pressure. It shows how stiff the patient's arteries are and how hard the heart must work to pump blood.


In [ ]:
# --- FEATURE 1: PULSE PRESSURE (Calculated using baseline ER admission vitals) ---
# This marker is carried forward across all days due to the cumulative nature of the system.

if "S_AD_ORIT" in X_train_d1_fe.columns and "D_AD_ORIT" in X_train_d1_fe.columns:
    X_train_d1_fe["PULSE_PRESSURE"] = X_train_d1_fe["S_AD_ORIT"] - X_train_d1_fe["D_AD_ORIT"]

if "S_AD_ORIT" in X_train_d2_fe.columns and "D_AD_ORIT" in X_train_d2_fe.columns:
    X_train_d2_fe["PULSE_PRESSURE"] = X_train_d2_fe["S_AD_ORIT"] - X_train_d2_fe["D_AD_ORIT"]

if "S_AD_ORIT" in X_train_d3_fe.columns and "D_AD_ORIT" in X_train_d3_fe.columns:
    X_train_d3_fe["PULSE_PRESSURE"] = X_train_d3_fe["S_AD_ORIT"] - X_train_d3_fe["D_AD_ORIT"]

print(f"✅ Pulse Pressure added. Shapes -> Day 1: {X_train_d1_fe.shape} | Day 2: {X_train_d2_fe.shape} | Day 3: {X_train_d3_fe.shape}")


✅ Pulse Pressure added. Shapes -> Day 1: (1360, 47) | Day 2: (1360, 59) | Day 3: (1360, 108)


# Step 5: Feature 2- Inflammatory Index

- **INFLAMMATORY_INDEX**: A combined measure of white blood cells and blood sedimentation rate. It reveals if the body is having a massive, dangerous immune reaction after the heart attack.


In [ ]:
# --- FEATURE 2: INFLAMMATORY_INDEX (Calculated strictly from Day 2 laboratory test results) ---
# Omitted from Day 1 to avoid time-based leakage, but carried forward dynamically into Day 3.

if "L_BLOOD" in X_train_d2_fe.columns and "ROE" in X_train_d2_fe.columns:
    X_train_d2_fe["INFLAMMATORY_INDEX"] = X_train_d2_fe["L_BLOOD"] * X_train_d2_fe["ROE"]

if "L_BLOOD" in X_train_d3_fe.columns and "ROE" in X_train_d3_fe.columns:
    X_train_d3_fe["INFLAMMATORY_INDEX"] = X_train_d3_fe["L_BLOOD"] * X_train_d3_fe["ROE"]

print(f"✅ Inflammatory Index added. Shapes -> Day 1: {X_train_d1_fe.shape} | Day 2: {X_train_d2_fe.shape} | Day 3: {X_train_d3_fe.shape}")


✅ Inflammatory Index added. Shapes -> Day 1: (1360, 47) | Day 2: (1360, 60) | Day 3: (1360, 109)


# Step 6: Feature 3- High-Risk Profile

- **HIGH_RISK_PROFILE**: A marker for the most endangered patients. It highlights elderly patients who waited too long to get to the hospital and arrived with life-threatening complications.



In [ ]:
# --- FEATURE 3: HIGH_RISK_PROFILE (A composite clinical score engineered specifically for Day 3 evaluation) ---
# Strictly isolated to Day 3 to evaluate late-stage patient frailty and admission delay.

if "AGE" in X_train_d3_fe.columns and "GB" in X_train_d3_fe.columns and "TIME_B_S" in X_train_d3_fe.columns:
    X_train_d3_fe["HIGH_RISK_PROFILE"] = ((X_train_d3_fe["AGE"] > 75) & (X_train_d3_fe["GB"] == 3) & (X_train_d3_fe["TIME_B_S"] > 4)).astype(int)

print(f"✅ High-Risk Profile added. Shapes -> Day 1: {X_train_d1_fe.shape} | Day 2: {X_train_d2_fe.shape} | Day 3: {X_train_d3_fe.shape}")


✅ High-Risk Profile added. Shapes -> Day 1: (1360, 47) | Day 2: (1360, 60) | Day 3: (1360, 110)


# Step 7: Feature 4- High-risk elderly interaction


- **ELDERLY_HYPERTENSION_RISK**: A combination of age and high blood pressure. It shows how dangerous chronic high blood pressure becomes when it damages the already fragile blood vessels of older patients.

In [ ]:
# Based on the EDA age-breakdown chart, patients over 70 exhibit an extreme surge in mortality.
# To capture this risk continuously without triggering One-Hot expansion in our preprocessing pipeline,
# we multiply the exact AGE by the chronic hypertension stage (GB).
# This creates a true high-variance continuous interaction term (unique values > 10).

print("⏳ Engineering 'ELDERLY_HYPERTENSION_RISK' continuous interaction across timelines...")

# --- 📅 DAY 1 FEATURE ENGINEERING ---
# Quantifies the exact age-weighted cardiovascular risk load upon ER admission
X_train_d1_fe["ELDERLY_HYPERTENSION_RISK"] = X_train_d1_fe["AGE"] * X_train_d1_fe["GB"]

# --- 📅 DAY 2 FEATURE ENGINEERING (Cumulative integration) ---
X_train_d2_fe["ELDERLY_HYPERTENSION_RISK"] = X_train_d2_fe["AGE"] * X_train_d2_fe["GB"]

# --- 📅 DAY 3 FEATURE ENGINEERING (Cumulative integration) ---
X_train_d3_fe["ELDERLY_HYPERTENSION_RISK"] = X_train_d3_fe["AGE"] * X_train_d3_fe["GB"]

print("✅ 'ELDERLY_HYPERTENSION_RISK' successfully engineered for Day 1, Day 2, and Day 3!")
print(f"Current Slices -> Day 1: {X_train_d1_fe.shape} | Day 2: {X_train_d2_fe.shape} | Day 3: {X_train_d3_fe.shape}")


⏳ Engineering 'ELDERLY_HYPERTENSION_RISK' continuous interaction across timelines...
✅ 'ELDERLY_HYPERTENSION_RISK' successfully engineered for Day 1, Day 2, and Day 3!
Current Slices -> Day 1: (1360, 48) | Day 2: (1360, 61) | Day 3: (1360, 111)


# Step 8: Lock changes into master variables


In [ ]:
# LOCK ENGINEERED COPIES BACK TO Master VARIABLES

X_train_d1 = X_train_d1_fe
X_train_d2 = X_train_d2_fe
X_train_d3 = X_train_d3_fe

print("=== 🎉 ALL SEPARATE FEATURE CELLS COMPLETED AND LOCKED! ===")
print(f"Engineered Raw Shapes -> Day 1: {X_train_d1.shape} | Day 2: {X_train_d2.shape} | Day 3: {X_train_d3.shape}")


=== 🎉 ALL SEPARATE FEATURE CELLS COMPLETED AND LOCKED! ===
Engineered Raw Shapes -> Day 1: (1360, 48) | Day 2: (1360, 61) | Day 3: (1360, 111)


# Step 9: Dynamic pipeline processing and quality control (Sanity check)


In [ ]:
print("⏳ Executing full pipeline (Imputation, Encoding, Scaling) on engineered data...")

# --- ⏳ RUNNING THE PIPELINE ON THE COMPLETE '_fe' DATASETS ---
# FIXED: Passing the engineered dataframes (_fe) which actually contain your new features!
X_train_d1_fe_ready = process_day_data(X_train_d1_fe, day_key="day1")
X_train_d2_fe_ready = process_day_data(X_train_d2_fe, day_key="day2")
X_train_d3_fe_ready = process_day_data(X_train_d3_fe, day_key="day3")

print("\n=== 🎉 Preprocessing and Scaling Completed Successfully! ===")

# --- INTEGRATED QUALITY CONTROL (SANITY CHECK) ---
nan_day1 = X_train_d1_fe_ready.isnull().sum().sum()
nan_day2 = X_train_d2_fe_ready.isnull().sum().sum()
nan_day3 = X_train_d3_fe_ready.isnull().sum().sum()

shape_d1 = X_train_d1_fe_ready.shape
shape_d2 = X_train_d2_fe_ready.shape
shape_d3 = X_train_d3_fe_ready.shape

print("\n--- 📋 Preprocessing Quality Control Report ---")
print(f"Missing Values left (Day 1 / Day 2 / Day 3): {nan_day1} / {nan_day2} / {nan_day3}")
print(f"Final Data Shapes  (Day 1 / Day 2 / Day 3): {shape_d1} / {shape_d2} / {shape_d3}")

is_nan_clean = (nan_day1 == 0 and nan_day2 == 0 and nan_day3 == 0)
# Note: shape_d1[0] checks the row count (1360)
are_rows_intact = (shape_d1[0] == 1360 and shape_d2[0] == 1360 and shape_d3[0] == 1360)
is_data_safe_to_save = is_nan_clean and are_rows_intact

if is_data_safe_to_save:
    print("\n🎉 Sanity Check Passed! Data size is 100% intact and completely imputed.")
    print("The engineered datasets are verified and safe for disk serialization.")
else:
    print("\n⚠️ Sanity Check Failed! Missing values or data loss detected. Review previous steps.")


⏳ Executing full pipeline (Imputation, Encoding, Scaling) on engineered data...

=== 🎉 Preprocessing and Scaling Completed Successfully! ===

--- 📋 Preprocessing Quality Control Report ---
Missing Values left (Day 1 / Day 2 / Day 3): 0 / 0 / 0
Final Data Shapes  (Day 1 / Day 2 / Day 3): (1360, 101) / (1360, 114) / (1360, 191)

🎉 Sanity Check Passed! Data size is 100% intact and completely imputed.
The engineered datasets are verified and safe for disk serialization.


# Step 10: Saving Final FE_READY Files to Drive

In [ ]:
if is_data_safe_to_save:
    folder_drive = "/content/drive/MyDrive/Mid-term Project/"

    print(f"⏳ Serializing and exporting engineered FE_ready datasets to: {folder_drive}")

    # FIXED: Changed index=False to index=True to permanently save patient IDs in the CSV file!
    X_train_d1_fe_ready.to_csv(os.path.join(folder_drive, "X_train_day1_FE_ready.csv"), index=True)
    X_train_d2_fe_ready.to_csv(os.path.join(folder_drive, "X_train_day2_FE_ready.csv"), index=True)
    X_train_d3_fe_ready.to_csv(os.path.join(folder_drive, "X_train_day3_FE_ready.csv"), index=True)

    print("\n=== 💾 ALL PROCESSED FEATURE-ENGINEERED DATASETS SAVED TO DRIVE! ===")


⏳ Serializing and exporting engineered FE_ready datasets to: /content/drive/MyDrive/Mid-term Project/

=== 💾 ALL PROCESSED FEATURE-ENGINEERED DATASETS SAVED TO DRIVE! ===


In [ ]:
# Verify that your engineered features exist in the final ready dataframes
print("Is PULSE_PRESSURE in Day 1 ready data?", "PULSE_PRESSURE" in X_train_d1_fe_ready.columns)
print("Is INFLAMMATORY_INDEX in Day 2 ready data?", "INFLAMMATORY_INDEX" in X_train_d2_fe_ready.columns)
print("Is HIGH_RISK_PROFILE in Day 3 ready data?", "HIGH_RISK_PROFILE" in X_train_d3_fe_ready.columns)
print("Is ELDERLY_HYPERTENSION_RISK in Day 1 ready data?", "ELDERLY_HYPERTENSION_RISK" in X_train_d1_fe_ready.columns)


Is PULSE_PRESSURE in Day 1 ready data? True
Is INFLAMMATORY_INDEX in Day 2 ready data? True
Is HIGH_RISK_PROFILE in Day 3 ready data? True
Is ELDERLY_HYPERTENSION_RISK in Day 1 ready data? True


## 🏁 Feature Engineering & Processing Pipeline Completed

### 📊 Summary of Executed Operations in this Notebook:
* **Clinical Feature Engineering (On Raw Scale)**: Successfully engineered 4 advanced clinical variables directly on raw numerical timelines, ensuring strict chronological isolation:
  * **Day 1**: `PULSE_PRESSURE` (ER blood pressure gap) and `ELDERLY_HYPERTENSION_RISK` (a continuous interaction term multiplying exact Age by chronic Hypertension stage, designed as a continuous feature to naturally protect its exact identity from One-Hot expansion).
  * **Day 2**: `INFLAMMATORY_INDEX` (amplifying the latent biological risk of post-infarction hyper-inflammation by multiplying L_BLOOD and ROE).
  * **Day 3**: `HIGH_RISK_PROFILE` (a compound clinical stability score combining age, extreme hypertension, and extensive ambulance hospital arrival delays).
* **Pipeline Processing & Quality Control**: Passed the raw engineered matrices through our index-safe preprocessing pipeline for median/mode imputation, multi-class One-Hot Encoding matching the patient rows, and StandardScaler feature regularization.
* **Sanity Check Validation**: Verified that all missing values (NaNs) across all timelines were completely resolved to 0, while perfectly preserving row indices for all 1,360 patients.
* **Background State Persistence**: Quietly logged all fitted imputer and scaler objects into a global tracking directory (`all_day_transformers`), ensuring zero inference-stage data leakage when processing the future Test Set.

---

### 💾 Generated CSV Files Exported to Google Drive (/Mid-term Project/):
All feature-engineered datasets have been explicitly exported with `index=True` to guarantee absolute alignment with target states.

* **Baseline Ready-to-Model Data from Notebook 1 (Without Feature Engineering):**
  * `X_train_day1_ready.csv` | `X_train_day2_ready.csv` | `X_train_day3_ready.csv`
* **Advanced Feature-Engineered Data from Notebook 2 (With Feature Engineering):**
  * `X_train_day1_FE_ready.csv` | `X_train_day2_FE_ready.csv` | `X_train_day3_FE_ready.csv`
* **Consolidated Multi-Label Targets:**
  * `y_train_ready.csv` (The final 5-column clinical outcome target matrix)

---

### 🚀 What Is the Next Step?
Modeling Notebook 4